In [ ]:
# Otomatis install pyserial jika belum terpasang
import sys
import subprocess

try:
    import serial
    import serial.tools.list_ports
except ImportError:
    print("[*] Installing pyserial module...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyserial"])
    import serial
    import serial.tools.list_ports

import time

# Configuration Parameters
SERIAL_BAUDRATE = 115200
STEP_SIZE = 10

# Hardware identifiers & descriptors
TARGET_VID_PID = {
    (0x10C4, 0xEA60), (0x10C4, 0xEA70),  # CP210x
    (0x1A86, 0x7523), (0x1A86, 0x55D4),  # CH340 / CH341
    (0x0403, 0x6001), (0x0403, 0x6015),  # FTDI
    (0x303A, 0x1001)                   # ESP32 Native USB
}

DRIVER_KEYWORDS = [
    "cp210", "cp2102", "cp210x", "ch340", "ch341", "ch34x",
    "ftdi", "ft232", "usb-serial", "usb serial", "usb uart",
    "uart bridge", "silicon labs", "espressif", "esp32"
]


def list_available_com_ports():
    ports = serial.tools.list_ports.comports()
    if not ports:
        print("[!] Tidak ada port serial yang terdeteksi.")
        return []

    print("=== Daftar Port Terdeteksi ===")
    for p in ports:
        hw_info = f" [VID:PID={p.vid:04X}:{p.pid:04X}]" if p.vid and p.pid else ""
        print(f" -> {p.device} ({p.description}){hw_info}")
    return ports


def auto_detect_esp32_port(verbose=True):
    active_ports = serial.tools.list_ports.comports()
    if not active_ports:
        if verbose:
            print("[X] Port serial tidak ditemukan.")
        return None

    scored_ports = []
    for port in active_ports:
        meta_str = f"{port.description or ''} {port.manufacturer or ''} {port.hwid or ''}".lower()
        score = 0

        # Prioritas berdasarkan ID Perangkat
        if (port.vid, port.pid) in TARGET_VID_PID:
            score += 120

        # Pencocokan kata kunci driver/chip
        if any(keyword in meta_str for keyword in DRIVER_KEYWORDS):
            score += 45

        # Penalti untuk perangkat Bluetooth
        if "bluetooth" in meta_str:
            score -= 250

        if score > 0:
            scored_ports.append((score, port.device, port.description))

    if not scored_ports:
        # Fallback: Cari port tunggal selain bluetooth
        non_bt_ports = [p.device for p in active_ports if "bluetooth" not in (p.description or "").lower()]
        if len(non_bt_ports) == 1:
            if verbose:
                print(f"[*] Menghubungkan ke satu-satunya port aktif: {non_bt_ports[0]}")
            return non_bt_ports[0]
        if verbose:
            print("[X] Tidak dapat menentukan port ESP32 secara otomatis.")
        return None

    # Urutkan berdasarkan skor tertinggi
    scored_ports.sort(key=lambda x: x[0], reverse=True)
    if verbose:
        print("\nHasil Pemindaian Port:")
        for sc, dev, desc in scored_ports:
            print(f"  - {dev} (Skor Match: {sc}) | {desc}")

    return scored_ports[0][1]


def establish_serial_connection(port_name, baud=SERIAL_BAUDRATE):
    try:
        connection = serial.Serial(port_name, baudrate=baud, timeout=1)
        time.sleep(2)  # Tunggu ESP32 reset/siap setelah penyambungan
        print(f"[+] Terhubung ke {port_name} ({baud} bps)")
        return connection
    except serial.SerialException as err:
        print(f"[-] Gagal membuka port {port_name}: {err}")
        return None


def send_command(ser_dev, command_payload: str):
    data_to_send = f"{command_payload}\n".encode("utf-8")
    ser_dev.write(data_to_send)
    ser_dev.flush()
    time.sleep(0.1)

    # Baca tanggapan dari ESP32
    while ser_dev.in_waiting > 0:
        raw_resp = ser_dev.readline().decode("utf-8", errors="ignore").strip()
        if raw_resp:
            print(f"[Respon ESP32] {raw_resp}")
        time.sleep(0.03)


# Fungsi Operasional Motor
def set_motor_state(ser_dev, state: bool):
    cmd = "1" if state else "0"
    action = "ON" if state else "OFF"
    print(f">> Kontrol Power: Motor {action}")
    send_command(ser_dev, cmd)


def adjust_speed(ser_dev, direction: str, delta=STEP_SIZE):
    if direction in ["+", "-"]:
        label = "Naik" if direction == "+" else "Turun"
        print(f">> Sesuaikan Kecepatan: {label} ({direction}{delta})")
        send_command(ser_dev, direction)


def set_direct_pwm(ser_dev, pwm_val: int):
    clamped_pwm = max(0, min(255, int(pwm_val)))
    print(f">> Set Nilai PWM: {clamped_pwm}")
    # Mengirim format "p180" atau "PWM:180" sesuai penanganan firmware ESP32
    send_command(ser_dev, f"p{clamped_pwm}")


# Alur Eksekusi Utama
if __name__ == "__main__":
    print("=" * 55)
    print("       SISTEM KONTROL MOTOR ESP32 VIA SERIAL")
    print("=" * 55)

    list_available_com_ports()
    print("-" * 55)

    selected_port = auto_detect_esp32_port()
    if not selected_port:
        print("[!] Program dihentikan: Port ESP32 tidak ditemukan.")
        sys.exit(1)

    esp32_serial = establish_serial_connection(selected_port)

    if esp32_serial:
        try:
            print("\n[ Run Skenario Otomatis ]")
            set_motor_state(esp32_serial, True)
            time.sleep(1.5)

            for _ in range(3):
                adjust_speed(esp32_serial, "+")
                time.sleep(0.7)

            for _ in range(2):
                adjust_speed(esp32_serial, "-")
                time.sleep(0.7)

            set_direct_pwm(esp32_serial, 180)
            time.sleep(2)

            set_motor_state(esp32_serial, False)
            print("[ Skenario Selesai ]\n")

            print("=== Panduan Input Manual ===")
            print("  1  : Nyalakan Motor")
            print("  0  : Matikan Motor")
            print("  +  : Tambah Kecepatan")
            print("  -  : Kurangi Kecepatan")
            print("  pX : Atur PWM langsung (misal: p150)")
            print("  q  : Keluar\n")

            while True:
                user_input = input("Perintah > ").strip()

                if user_input.lower() == "q":
                    break
                elif user_input == "1":
                    set_motor_state(esp32_serial, True)
                elif user_input == "0":
                    set_motor_state(esp32_serial, False)
                elif user_input in ["+", "-"]:
                    adjust_speed(esp32_serial, user_input)
                elif user_input.lower().startswith("p") and user_input[1:].isdigit():
                    set_direct_pwm(esp32_serial, int(user_input[1:]))
                else:
                    print("[!] Perintah tidak valid.")

        except KeyboardInterrupt:
            print("\n[!] Dihentikan oleh pengguna.")

        finally:
            try:
                set_motor_state(esp32_serial, False)
            except Exception:
                pass
            esp32_serial.close()
            print("[+] Port serial berhasil ditutup.")

       SISTEM KONTROL MOTOR ESP32 VIA SERIAL
=== Daftar Port Terdeteksi ===
 -> COM6 (USB-Enhanced-SERIAL CH9102 (COM6)) [VID:PID=1A86:55D4]
-------------------------------------------------------

Hasil Pemindaian Port:
  - COM6 (Skor Match: 120) | USB-Enhanced-SERIAL CH9102 (COM6)
[+] Terhubung ke COM6 (115200 bps)

[ Run Skenario Otomatis ]
>> Kontrol Power: Motor ON
>> Sesuaikan Kecepatan: Naik (+10)
[Respon ESP32] Motor ON | PWM=200 | RPM=0.0
>> Sesuaikan Kecepatan: Naik (+10)
[Respon ESP32] PWM NAIK | PWM=210 | RPM=0.0
>> Sesuaikan Kecepatan: Naik (+10)
[Respon ESP32] PWM NAIK | PWM=220 | RPM=0.0
>> Sesuaikan Kecepatan: Turun (-10)
[Respon ESP32] PWM NAIK | PWM=230 | RPM=0.0
>> Sesuaikan Kecepatan: Turun (-10)
[Respon ESP32] PWM TURUN | PWM=220 | RPM=0.0
>> Set Nilai PWM: 180
[Respon ESP32] PWM TURUN | PWM=210 | RPM=0.0
>> Kontrol Power: Motor OFF
[ Skenario Selesai ]

=== Panduan Input Manual ===
  1  : Nyalakan Motor
  0  : Matikan Motor
  +  : Tambah Kecepatan
  -  : Kurangi K